# Assignment 2

Deadline: 25.03.2025, 12:00 CET

<Add your name, student-id and emal address>

## 1. Linearization of Turnover

**(15 points)**

Turnover constraints are used to limit the amount of change in portfolio weights between periods, helping to manage transaction costs and maintain portfolio stability.

Your task is to implement a method `linearize_turnover_constraint` for the class `QuadraticProgram`, which modifies the quadratic programming problem to incorporate a linearized turnover constraint. This will involve updating the objective function coefficients, equality and inequality constraints, as well as the lower and upper bounds of the problem. 

Additionally, complete the example provided below to demonstrate that your method functions correctly.

In class, we discussed a solution that involved augmenting the dimensionality by a factor of three. Here, you are asked to implement an alternative method that involves a two-fold increase in dimensions. If you are unable to implement the two-fold method, you may proceed with the three-fold approach.

### Function Parameters:
- `x_init` (np.ndarray): The initial portfolio weights.
- `to_budget` (float, optional): The maximum allowable turnover. Defaults to `float('inf')`.

### Steps for Function Implementation:

As discussed in the lecture, introduce auxiliary variables and augment the matrices/vectors used for optimization.

- **Objective Function Coefficients**:  
  Pad the existing objective function coefficients (`P` and `q`) to accommodate the new variables introduced by the turnover constraint.  
  *Note*: "Padding" refers to adding extra elements (typically zeros) to an array or matrix to increase its size to a desired shape.

- **Equality Constraints**:  
  Pad the existing equality constraint matrix (`A`) to account for the new variables.

- **Inequality Constraints**:  
  Pad the existing inequality constraint matrix ('G') and vector ('h') and further add a new inequality constraint row to incorporate the turnover constraint.  

- **Lower and Upper Bounds**:  
  Pad the existing lower (`lb`) and upper (`ub`) bounds to accommodate the new variables.

- **Update Problem Data**:  
  Overwrite the original problem data in the `QuadraticProgram` class with the updated matrices and vectors to include the linearized turnover constraint.

In [38]:
# Import standard libraries
import types
import os
import sys

# Import third-party libraries
import numpy as np
import pandas as pd

# Import local modules
project_root = '/Users/nawangpegentsang/Desktop/'  # Change this path if needed
src_path = os.path.join(project_root, 'qpmwp-course/src')
sys.path.append(project_root)
sys.path.append(src_path)
print(f"Project root added to sys.path: {project_root}")
print(f"Source path added to sys.path: {src_path}")
from estimation.covariance import Covariance
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.quadratic_program import QuadraticProgram
from helper_functions import load_data_msci

Project root added to sys.path: /Users/nawangpegentsang/Desktop/
Source path added to sys.path: /Users/nawangpegentsang/Desktop/qpmwp-course/src


In [39]:
def linearize_turnover_constraint(self, x_init: np.ndarray, to_budget=float('inf')) -> None:
        '''
        Linearize the turnover constraint in the quadratic programming problem.

        This method modifies the quadratic programming problem to include a linearized turnover constraint.

        Parameters:
        -----------
        x_init : np.ndarray
            The initial portfolio weights.
        to_budget : float, optional
            The maximum allowable turnover. Defaults to float('inf').

        Notes:
        ------
        - The method updates the problem's objective function coefficients, inequality constraints,
        equality constraints, and bounds to account for the turnover constraint.
        - The original problem data is overridden with the updated matrices and vectors.

        Examples:
        ---------
        >>> qp = QuadraticProgram(P, q, G, h, A, b, lb, ub, solver='cvxopt')
        >>> qp.linearize_turnover_constraint(x_init=np.array([0.1, 0.2, 0.3]), to_budget=0.05)
        '''
        # Dimensions
        n = len(self.problem_data.get('q')) # gives us number of assets
        m = 0 if self.problem_data.get('G') is None else self.problem_data.get('G').shape[0] # gives us number of inequality constraints

        # Update the coefficients of the objective function
        P0 = self.problem_data.get('P')
        q0 = self.problem_data.get('q')

        P = np.zeros((2*n, 2*n)) # New P should be (2n x 2n), with P0 in the top-left corner
        P[:n, :n] = P0
        
        q = np.zeros(2*n) # New q should be (2n,), with q0 in the first n elements
        q[:n] = q0

        # Update the equality constraints
        A0 = self.problem_data.get('A')
        A = None if A0 is None else np.hstack((A0, np.zeros((A0.shape[0], n)))) # hstack: horizontally (column-wise) stack A0 with a zero matrix of shape (A0.shape[0], n)

        # Update the inequality constraints
        G0 = self.problem_data.get('G')
        h0 = self.problem_data.get('h')

        # New G has m + 2n + 1 rows and 2n columns
        G = np.zeros((m + 2*n + 1, 2*n))
        h = np.zeros(m + 2*n + 1)

        # Block 1: original constraints (first m rows)
        if G0 is not None and h0 is not None:
            G[:m, :n] = G0
            h[:m] = h0

        # Block 2: w - t <= x_init (next n rows)
        G[m:m+n, :n] = np.eye(n) # Identity matrix of size n for the w variables
        G[m:m+n, n:] = -np.eye(n) # Negative identity matrix of size n for the t variables
        h[m:m+n] = x_init # x_init vector for the right-hand side
        
        # Block 3: -w - t <= -x_init (next n rows)
        G[m+n:m+2*n, :n] = -np.eye(n) # Negative identity matrix of size n for the w variables
        G[m+n:m+2*n, n:] = -np.eye(n) # Negative identity matrix of size n for the t variables
        h[m+n:m+2*n] = -x_init # -x_init vector for the right-hand side

        # Block 4: sum(t) <= T (last row)   
        G[m+2*n, n:] = np.ones(n) # Vector of ones for the t variables
        h[m+2*n] = to_budget # to_budget scalar for the right-hand side

        # Update lower and upper bounds
        lb0 = self.problem_data.get('lb')
        ub0 = self.problem_data.get('ub')

        # Need to handle None case for both lb and ub
        lb = np.zeros(2*n) if lb0 is None else np.hstack((lb0, np.zeros(n)))
        ub = np.full(2*n, np.inf) if ub0 is None else np.hstack((ub0, np.full(n, np.inf)))
        # np.full(2*n, np.inf) creates an array of shape (2n,) filled with np.inf

        # Override the original matrices (notice: b does not change)
        self.update_problem_data({
            'P': P,
            'q': q,
            'G': G,
            'h': h,
            'A': A,
            'lb': lb,
            'ub': ub,
        })
        return None

## Demo

#### Create P and q

In [40]:
# Load the msci country index data
N = 10
data = load_data_msci(path = '/Users/nawangpegentsang/Desktop/qpmwp-course/data/', n=N)
X = data['return_series']
print(X.head()) # Display the first few rows of the return series to verify it loaded correctly

# Compute the vector of expected returns (mean returns)
q = ExpectedReturn(method='geometric').estimate(X=X, inplace=False)

# Compute the covariance matrix
P = Covariance(method='pearson').estimate(X=X, inplace=False)


                  AT        AU        BE       CA        CH        DE  \
Index                                                                   
1999-01-01  0.000000  0.000000  0.000000  0.00000  0.000000  0.000000   
1999-01-04  0.010057  0.009080  0.042147  0.01307  0.035885  0.052249   
1999-01-05  0.013661 -0.010048  0.020162  0.02194  0.012016  0.001444   
1999-01-06  0.000000  0.015264 -0.000078  0.02764  0.015335  0.036205   
1999-01-07  0.004104  0.016564 -0.016877 -0.00348 -0.011902 -0.020187   

                  DK        ES        FI        FR  
Index                                               
1999-01-01  0.000000  0.000000  0.000000  0.000000  
1999-01-04  0.026198  0.069051  0.052778  0.049229  
1999-01-05 -0.001789  0.026011  0.014154  0.011346  
1999-01-06  0.000432  0.000000  0.000000  0.021537  
1999-01-07 -0.019041 -0.015610  0.028012 -0.013856  


### Create some constraints, instantiate an object of class QuadraticProgram, and add the method linearize_turnover_constraint to the instance.

In [47]:
# Instantiate the constraints with only the budget and long-only constraints
constraints = Constraints(ids = X.columns.tolist()) # Creates constraint object with ids 
constraints.add_budget(rhs=1, sense='=')
constraints.add_box(lower=0.0, upper=1.0)
GhAb = constraints.to_GhAb()

# Create a quadratic program and linearize the turnover constraint
qp = QuadraticProgram(
    P = P.to_numpy(),
    q = q.to_numpy() * 0,
    G = GhAb['G'],
    h = GhAb['h'],
    A = GhAb['A'],
    b = GhAb['b'],
    lb = constraints.box['lower'].to_numpy(),
    ub = constraints.box['upper'].to_numpy(),
    solver = 'cvxopt',
)

# Add the linearized turnover constraint method to the instance of class QuadraticProgram
qp.linearize_turnover_constraint = types.MethodType(linearize_turnover_constraint, qp) # attach the method to the instance of class QuadraticProgram

#types.MethodType() does two things: it binds the function linearize_turnover_constraint to the instance qp, and it allows us to call this function as a method of qp, meaning we can use self inside the function to refer to qp's attributes and methods.

### Add a turnover limit of 50%. Solve the problem and check whether the turnover constraint is respected.

In [ ]:
# Prepare initial weights
x_init = pd.Series([1/X.shape[1]]*X.shape[1], index=X.columns) # Initial weights (equal-weighted portfolio)

# Add the linearized turnover constraint
qp.linearize_turnover_constraint(x_init=x_init, to_budget=0.5) 

# Check the updated problem data dimensions (should be (2N, 2N))
print(qp.problem_data['P'].shape)

# Solve the problem
qp.solve()

# Check the turnover
solution = qp.results.get('solution')
ids = constraints.ids
weights = pd.Series(solution.x[:len(ids)], index=ids)
# note after adding the turnover constraint, the optimizer contains both the original weights and the auxiliary variables for turnover, but we only want to look at the original weights to compute turnover against x_init

print("Turnover:")
print(np.abs(weights - x_init).sum())


(20, 20)
Turnover:
0.499545543098582
